# Security Tooling for Developers

A leaked API key in git history is practically permanent once pushed. GitHub indexes commit history, and forks preserve deleted content — rotating the key is the only reliable fix. The two distinct threat classes are (1) *secrets in code*, where credentials end up in committed files or commit history, and (2) *vulnerable dependencies*, where a package already in the lockfile is later found to have a known CVE. These require different defenses.

This notebook documents a defense-in-depth stack for Python projects addressing both: (1) a pre-commit hook catches secrets before they leave the machine, (2) dependency vulnerability scanning triggers when dependencies change, (3) CI gates block secrets and vulnerable packages on every pull request, and (4) runtime secret management prevents secrets from entering source files in the first place. Additional topics covered include supply chain attack defense using `uv`'s lockfile integrity model, workflow hardening via SHA-pinned Actions and `zizmor` auditing, and static analysis with Bandit for Python-specific security issues. The appendix covers recovery from a leak.

All tooling discussed targets the Python ecosystem specifically: `uv` for dependency management, `pip-audit` against the PyPA and GitHub Advisory Databases, Bandit for Python AST analysis, and `python-dotenv` for runtime secret loading.

## Pre-commit Secrets Scanner

The primary tool is **Gitleaks** — a fast, Go-based scanner with built-in rules for over 150 secret types (OpenAI keys, AWS credentials, GitHub tokens, etc.). For teams needing a Python-installable alternative or large false-positive baseline management, **detect-secrets** is an option, but Gitleaks is preferred here for its zero-Python-dependency installation and broad default ruleset.

**Install.** Add `pre-commit` as a dev dependency and register the hook:

```{.bash filename="$ (local)"}
uv add --dev pre-commit
pre-commit install
```

`pre-commit` manages hook environments in isolation — it downloads and installs Gitleaks automatically via the hook config, so no separate binary installation is needed. The hook runs on every `git commit`.

**Configure.** The `.pre-commit-config.yaml` file at the repo root declares which hooks to run:

```{.yaml filename=".pre-commit-config.yaml"}
repos:
  - repo: https://github.com/gitleaks/gitleaks
    rev: v8.24.2
    hooks:
      - id: gitleaks
```

Pin `rev` to a specific release tag rather than `main` — mutable branch refs mean the hook silently changes behavior on the next `pre-commit autoupdate`.

**Run (audit existing repo).** `pre-commit run gitleaks --all-files` scans all tracked files, not just staged ones. This is useful for an initial audit of an existing repository:

```{.bash filename="$ (local)"}
pre-commit run gitleaks --all-files
```

A finding looks like:

```
Finding:     secret_key="sk-proj-REDACTED..."
Secret:      sk-proj-REDACTED...
RuleID:      openai-api-key
Entropy:     4.12
File:        notebooks/agents/01.ipynb
Line:        42
Fingerprint: notebooks/agents/01.ipynb:openai-api-key:42
```

The `Fingerprint` field is a stable identifier for the finding — it is used to permanently suppress false positives.

**False positives.** Three mechanisms handle legitimate strings that resemble secrets: (1) inline suppression via `# gitleaks:allow` on the offending line, (2) a `.gitleaksignore` file containing fingerprints to suppress permanently, and (3) a custom `.gitleaks.toml` for project-wide allow-listing of paths or patterns.

A minimal `.gitleaks.toml` extending the default ruleset:

```{.toml filename=".gitleaks.toml"}
[extend]
useDefault = true

[allowlist]
paths = [
    "tests/fixtures/",
]
```

The `useDefault = true` line ensures all built-in rules remain active — the config only adds to them.

:::{.callout-caution}
Do not habitually skip the hook with `SKIP=gitleaks git commit`. Use it once for a confirmed false positive, then immediately add the fingerprint to `.gitleaksignore` so the suppression is tracked in version control.

:::

## Dependency Vulnerability Checking

**pip-audit** queries the Python Packaging Advisory Database (PyPA) and the GitHub Advisory Database for CVEs in installed packages. **Dependabot** complements it by automatically opening pull requests to upgrade vulnerable packages. The two tools are used together: Dependabot fixes the problem proactively, pip-audit in CI enforces the fix is deployed before code ships.

**Timing trade-offs.** Where to run pip-audit involves a friction/freshness trade-off:

| Option | Dev friction | Freshness | Recommendation |
|---|---|---|---|
| Pre-commit hook | High — blocks every commit | At commit time | Avoid — network-bound, slows all commits |
| Manual after `uv add` | None | At dep change | Good habit; run `pip-audit --locked .` |
| Scheduled CI workflow | None | Weekly | Best for background scanning |
| Dependabot | None | Near-real-time | Best for automated upgrade PRs |

The recommended approach is to add Dependabot for automated security PRs, add a scheduled CI workflow for periodic audits, and run `pip-audit --locked .` manually after `uv add`. Do NOT add pip-audit as a pre-commit hook.

**Install and run locally.**

```{.bash filename="$ (local)"}
uv tool install pip-audit
```

Useful invocations:

```{.bash filename="$ (local)"}
pip-audit --locked .                                      # audit the locked dependency tree
pip-audit --locked . --desc --aliases                    # include CVE descriptions and aliases
pip-audit --locked . --ignore-vuln GHSA-w596-4wvx-j9j6  # suppress a known false positive
```

Advisory IDs use three prefixes: `PYSEC-` (PyPI Advisory Database), `CVE-` (NIST National Vulnerability Database), and `GHSA-` (GitHub Advisory Database). A finding looks like:

```
Found 1 known vulnerability in 1 package
Name    Version ID                  Fix Versions
------- ------- ------------------- ------------
pillow  9.0.0   GHSA-56pw-mpj4-fxjw 9.0.1
```

:::{.callout-note}
`--locked` audits `uv.lock`, not the activated environment — it catches vulnerabilities before `uv sync` installs them. Omitting `--locked` audits the live environment instead, which may differ from the lockfile.

:::

**Dependabot.** Enable automated dependency upgrade PRs by adding `.github/dependabot.yml`:

```{.yaml filename=".github/dependabot.yml"}
version: 2
updates:
  - package-ecosystem: "uv"
    directory: "/"
    schedule:
      interval: "weekly"
    groups:
      patch-and-minor:
        update-types: ["patch", "minor"]

  - package-ecosystem: "github-actions"
    directory: "/"
    schedule:
      interval: "weekly"
```

Notice that `package-ecosystem: "uv"` is the correct value for `uv`-managed projects — not `"pip"`. The `groups` block batches patch and minor upgrades into a single PR rather than one PR per package, reducing noise.

Do not enable auto-merge on Dependabot PRs. The `uv.lock` diff in each PR shows exactly which version changed and what the new hashes are — reviewing it is the human gate that makes a supply chain compromise visible before it ships.

## Supply Chain Attack Defense

CVE-based scanning (pip-audit, Dependabot) catches vulnerabilities that have already been disclosed and indexed in an advisory database. Supply chain attacks arrive through a different vector: a legitimate, well-known package is compromised *before* any CVE exists. The typical scenario is a compromised maintainer account — an attacker gains PyPI credentials and publishes a new patch or minor release that passes all name and signature checks. Automated CI that pulls the latest matching version installs it silently, and advisory databases are empty at the time of installation. pip-audit will not flag this. The defense operates at the artifact level rather than the advisory level.

**How `uv` provides integrity guarantees.** The `uv.lock` file records the exact version, download URL, SHA-256 hash, and byte size of every package artifact — one entry per wheel variant per platform, plus the sdist. A representative entry looks like:

```toml
[[package]]
name = "litellm"
version = "1.63.2"
source = { registry = "https://pypi.org/simple" }
sdist = { url = "https://files.pythonhosted.org/...", hash = "sha256:a1b2c3...", size = 5823104 }
wheels = [
    { url = "https://files.pythonhosted.org/...", hash = "sha256:d4e5f6...", size = 6012287 },
]
```

When `uv sync --frozen` runs, it downloads the artifact and verifies the hash before writing anything to the environment. If the file served by PyPI does not match the recorded SHA-256 — even for the same version string — the install fails immediately. This is the artifact-level defense: the version is pinned and the file content is pinned independently.

**`--frozen` in CI.** The default `uv sync` re-resolves the dependency graph on each run, which can pull in a newer compatible release if the lockfile is absent or stale. In CI, always use `--frozen`:

```{.bash filename="$ (CI)"}
uv sync --frozen
```

`--frozen` refuses to update `uv.lock` and raises an error if the lockfile is missing or inconsistent with `pyproject.toml`. Combined with the hash verification above, this means a compromised release can only enter the environment if someone first runs `uv lock --upgrade-package litellm` locally and commits the resulting lockfile diff — a deliberate, reviewable action.

**The upgrade gate.** The practical consequence is that upgrades become explicit and reviewable:

```{.bash filename="$ (local)"}
uv lock --upgrade-package litellm   # bump one package, regenerate lockfile
git diff uv.lock                    # review: new version, new hashes
```

The `uv.lock` diff shows exactly which version changed and what the new hashes are. This is the point at which a supply chain attack becomes visible — a Dependabot PR for a patch bump from a popular package during an active incident will show an unexpected hash change. Reviewing lockfile diffs on Dependabot PRs (rather than auto-merging them) is the human gate that complements the technical controls.

:::{.callout-caution}
Do not configure Dependabot with `auto-merge: true` for packages that have write access to production systems or CI secrets. Auto-merging removes the human review step that makes the lockfile diff visible.

:::

**Socket.dev: behavioral scanning before CVEs exist.** `socket.dev` analyzes each new release at publish time and flags behavioral anomalies: a new network call added to an install script, a new maintainer added to the project, new use of `exec()` or `subprocess` that was not in the previous version. These signals appear within minutes of a malicious release, before any CVE is filed. Install the [Socket GitHub App](https://github.com/apps/socket-security) to automatically scan every Dependabot PR. When a PR bumps a package to a flagged release, Socket posts a comment with the specific anomalies detected. No configuration beyond app installation is required.

The division of responsibility between the three tools is: pip-audit catches *known-bad* packages (CVE exists), Socket catches *newly-suspicious* packages (behavior changed), and `uv sync --frozen` ensures *only the reviewed version* is installed.

## CI Security Checks

Two CI workflows enforce security gates on every push and pull request: a secrets scan using `gitleaks/gitleaks-action@v2` and a dependency audit using `pypa/gh-action-pip-audit@v1`.

**Secrets scan on every push and PR.** The following workflow runs Gitleaks against the full commit history on every push to any branch and every pull request:

```{.yaml filename=".github/workflows/security-secrets.yml"}
name: Secrets Scan

on:
  push:
    branches: ["**"]
  pull_request:
    branches: ["**"]

jobs:
  gitleaks:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
        with:
          fetch-depth: 0           # <1>
      - uses: gitleaks/gitleaks-action@v2
        env:
          GITHUB_TOKEN: ${{ secrets.GITHUB_TOKEN }}
```


1. Gitleaks scans git history — full history requires `fetch-depth: 0`. The default shallow clone misses all commits beyond the latest, allowing a secret introduced in an older commit to pass undetected.

For personal repositories, no `GITLEAKS_LICENSE` secret is needed. GitHub organization repositories require a license key set as a repository secret.

**Scheduled dependency audit.** The following workflow runs pip-audit on a weekly schedule and also triggers immediately when the lockfile or manifest changes:

```{.yaml filename=".github/workflows/security-audit.yml"}
name: Dependency Audit

on:
  schedule:
    - cron: "0 9 * * 1"           # <1>
  push:
    paths:
      - "uv.lock"
      - "pyproject.toml"          # <2>

jobs:
  pip-audit:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: astral-sh/setup-uv@v7
      - uses: pypa/gh-action-pip-audit@v1.1.0
        with:
          inputs: "."
          args: "--locked"
```


1. Runs every Monday at 09:00 UTC regardless of commits — catches new CVEs disclosed since the last code change.
2. Also triggers immediately on any push that modifies the dependency lockfile or manifest, so a `uv add` that introduces a known vulnerability fails in CI before merging.

In any workflow that installs the project before running code, use `uv sync --frozen` rather than `uv sync`. The `--frozen` flag refuses to re-resolve the dependency graph and verifies the SHA-256 hash of every downloaded artifact against `uv.lock` before writing to the environment. Replace the plain checkout-and-sync pattern with:

```yaml
- uses: actions/checkout@v4
- uses: astral-sh/setup-uv@v7
- run: uv sync --frozen   # verifies hashes; fails if lockfile is stale
```

:::{.callout-note}
Dependabot creates PRs to upgrade vulnerable packages — it is asynchronous and proactive. pip-audit in CI blocks a merge if a vulnerability exists at that moment. Both are necessary: Dependabot surfaces the fix, pip-audit enforces that the fix is deployed before code ships.

:::

## GitHub Actions Workflow Hardening

The supply-chain risk in GitHub Actions is subtle: `uses: actions/checkout@v4` pins to a tag, but tags are mutable pointers — a compromised maintainer account can update `v4` to point to a different, malicious commit. Pinning to a full commit SHA eliminates this vector entirely:

```yaml
- uses: actions/checkout@11bd71901bbe5b1630ceea73d27597364c9af683  # v4.2.2
```

The inline comment preserves human readability while the SHA provides immutability. Tools like `zizmor` and Dependabot's `github-actions` ecosystem can automate the maintenance of SHA pins.

:::{.callout-important}
Third-party Actions execute arbitrary code in the CI runner with access to `secrets.GITHUB_TOKEN` and any secrets loaded into the job. A malicious action can exfiltrate credentials or push code to the repository. Pin to commit SHAs for any action in a security-sensitive workflow.

:::

**Install and run zizmor.**

```{.bash filename="$ (local)"}
uv tool install zizmor
zizmor .github/workflows/
```

`zizmor` audits workflow YAML files for: unpinned action refs, over-permissioned `permissions:` blocks, `pull_request_target` misuse (a common privilege escalation vector), and expression injection vulnerabilities where untrusted input like `github.event.pull_request.title` flows into a `run:` step.

Example output:

```
.github/workflows/publish.yml:
  1 finding(s):

  ⚠ unpinned-uses [medium]
    Line 17: uses: quarto-dev/quarto-actions/setup@v2
    Fix: pin to commit SHA
```

**CODEOWNERS.** Requiring a code-owner review for any workflow file change prevents an unauthorized contributor from adding a malicious step to an existing workflow:

```{.bash filename=".github/CODEOWNERS"}
.github/workflows/  @your-github-username
```

With this in place, a pull request touching any file under `.github/workflows/` cannot be merged without an approving review from the listed owner, even if the repository allows auto-merge for other PRs.

**Permissions: principle of least privilege.** By default, GitHub Actions grants the `GITHUB_TOKEN` `read` access to all scopes. Scope permissions at the job level so that only the job that needs write access gets it:

```yaml
jobs:
  build-deploy:
    permissions:
      contents: write      # only this job needs write; others default to read
```

Setting `permissions` at the workflow level with a permissive default and overriding per-job is less safe than starting from a restrictive default. A top-level `permissions: read-all` with explicit job-level grants is the recommended structure.

## Static Code Security Analysis

**Bandit** analyzes Python source for common security issues: hardcoded passwords (`B105`–`B107`), shell injection via `subprocess` with `shell=True` (`B602`/`B603`), use of `assert` for authentication checks (`B101`), and unsafe deserialization via `pickle` (`B301`). It is a static analysis tool — it does not execute code.

**Install and run.**

```{.bash filename="$ (local)"}
uv add --dev bandit
bandit -r src/ -ll        # medium and high severity only
```

The `-ll` flag filters to medium and high severity, reducing noise from low-severity informational findings during initial setup.

**Configure in `pyproject.toml`.**

```{.toml filename="pyproject.toml"}
[tool.bandit]
exclude_dirs = ["tests", "notebooks"]
skips = ["B101"]          # assert statements are acceptable in tests
```

Notebooks are excluded because they routinely use patterns that Bandit flags — hardcoded example values, `subprocess` calls for shell demos, and `pickle` for model checkpointing. Bandit is most useful scoped to `src/`.

**Pre-commit hook.**

```{.yaml filename=".pre-commit-config.yaml"}
- repo: https://github.com/PyCQA/bandit
  rev: 1.8.3
  hooks:
    - id: bandit
      args: ["-c", "pyproject.toml"]
```

The `-c pyproject.toml` argument ensures the hook respects the `[tool.bandit]` configuration above, including the `exclude_dirs` list.

**The pickle risk.** `pickle.load()` executes arbitrary Python code embedded in the serialized file. Loading a model checkpoint from an untrusted source is equivalent to running the file author's code with local filesystem permissions. The `weights_only=True` parameter in `torch.load` restricts deserialization to tensor data only, blocking code execution:

In [ ]:
import torch

# Unsafe: arbitrary code execution if the file is malicious
# model_state = torch.load("checkpoint.pt")

# Safe: weights_only=True restricts deserialization to tensors only
model_state = torch.load("checkpoint.pt", weights_only=True)

:::{.callout-caution}
`pickle.load(f)` will execute any Python code embedded in the file. Loading a Hugging Face checkpoint or a downloaded notebook model without auditing is equivalent to running the model author's code with local permissions. Use `weights_only=True` with `torch.load`, or switch to `safetensors` for model weights.

:::

## Runtime Secret Management

API keys for OpenAI, Groq, AWS, and similar services must never appear in notebook source, `pyproject.toml`, or any committed file. The standard pattern is a `.env` file on disk, gitignored, loaded at runtime via `python-dotenv`.

**Install.**

```{.bash filename="$ (local)"}
uv add python-dotenv
```

**`.env` file format** (never committed — add `.env` to `.gitignore`):

```{.bash filename=".env"}
OPENAI_API_KEY=sk-proj-...
GROQ_API_KEY=gsk_...
AWS_ACCESS_KEY_ID=AKIA...
AWS_SECRET_ACCESS_KEY=...
```

Commit a `.env.example` file alongside it with the same keys but empty values — this documents what variables are expected without exposing real credentials:

```{.bash filename=".env.example"}
OPENAI_API_KEY=
GROQ_API_KEY=
AWS_ACCESS_KEY_ID=
AWS_SECRET_ACCESS_KEY=
```

Verify that `.env` is gitignored with:

```{.bash filename="$ (local)"}
git check-ignore -v .env
```

If the command returns the path (e.g. `.gitignore:5:.env`), git will not track it. No output means it is not ignored.

**Load at runtime.** Call `load_dotenv()` at the top of any notebook or script that needs credentials:

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads .env from the working directory

api_key = os.environ["OPENAI_API_KEY"]  # raises KeyError if missing

`os.environ["KEY"]` raises `KeyError` if the variable is missing — it fails loudly at startup rather than silently failing later with an unhelpful error. `os.getenv("KEY", "default")` silently falls back to a hardcoded value, which is the wrong behavior for secrets: a misconfigured environment would proceed with a wrong or empty key.

**GitHub Actions secrets.** In CI, `.env` does not exist — secrets are injected via the GitHub Actions secret store and surfaced as environment variables in the job:

```yaml
- name: Run notebook
  env:
    OPENAI_API_KEY: ${{ secrets.OPENAI_API_KEY }}
  run: jupyter nbconvert --to notebook --execute notebook.ipynb
```

GitHub Actions automatically masks secret values in log output — any log line containing the raw secret value is replaced with `***`.

:::{.callout-note}
`.env` files are plaintext on disk — anyone with filesystem access can read them. They are appropriate for local development only. For shared environments, use the platform's secret store: GitHub Actions secrets, AWS Secrets Manager, 1Password CLI, or similar.

:::

## Appendix: Evaluating Untrusted Dependencies

The supply chain defenses covered above — pip-audit, socket.dev, `uv sync --frozen` — operate on packages that are *already in the lockfile*. They say nothing about whether a package should be added in the first place. In enterprise settings the question frequently arises with low-star or early-stage packages: a useful utility with 40 stars on GitHub, a specialized ML library from an unfamiliar author, a private registry package from a vendor. The following checklist structures the pre-adoption decision.

### Repository Health Signals

Star count is a weak signal in isolation — many useful enterprise utilities have few stars, and stars can be purchased. A more reliable picture comes from reading the commit and issue history:

| Signal | What to look for |
|---|---|
| Commit recency | Last commit within 6 months for active projects; stale repos may have unpatched CVEs |
| Contributor count | Single-maintainer projects carry key-person risk; check if the maintainer is responsive |
| Issue / PR responsiveness | Open bug reports unanswered for months, or closed with "won't fix" on security issues, are red flags |
| Release cadence | Erratic or sudden burst of releases after a long gap can indicate account compromise |
| Publishing history on PyPI | Visit `pypi.org/project/<name>/#history` — a package that published 20 releases in one week is suspicious |
| Maintainer account age | New GitHub/PyPI accounts publishing high-value package names are a typosquatting signal |

For private index packages (custom `--extra-index-url`), the same checks apply to the host organization's public presence — GitHub org creation date, number of public repos, any prior security disclosures.

### Source Code and Dependency Audit

Before adding a package, read the code that runs at install time and at import time. Two specific vectors:

**Install-time execution.** `setup.py` and `pyproject.toml` build backends can execute arbitrary code during `pip install`. Inspect these files for network calls, file writes outside the package directory, or `subprocess` calls:

```{.bash filename="$ (local)"}
# Download the sdist without installing it
pip download --no-deps --no-binary :all: programasweights \
    --extra-index-url https://pypi.programasweights.com/simple/ \
    -d /tmp/paw-dist

# Unpack and inspect
tar xzf /tmp/paw-dist/programasweights-*.tar.gz -C /tmp/paw-src
grep -r 'subprocess\|urllib\|requests\|socket\|eval\|exec' /tmp/paw-src/
```

**Transitive dependencies.** A small package can pull in a large dependency graph. Before committing the lockfile, review what `uv add` resolves:

```{.bash filename="$ (local)"}
# Dry-run: show what would be added without writing anything
uv add --dry-run programasweights \
    --extra-index-url https://pypi.programasweights.com/simple/
```

Inspect each new transitive package the same way. A two-function utility that pulls in `requests`, `boto3`, and `cryptography` as hard dependencies deserves scrutiny — the attack surface is the union of all transitive packages.

**Runtime behavior at import.** Some packages execute significant logic on `import`. After sandboxed installation (see below), run:

```{.bash filename="$ (sandbox)"}
python -c "import programasweights"  # observe: network calls? file writes?
```

Use a network-monitoring tool (`tcpdump`, `lsof`, or `strace -e trace=network` on Linux) to observe whether the import makes outbound connections.

### Sandboxing First Runs

The first install and first import of an untrusted package should happen in an isolated environment, not on the development machine. Three escalating levels of isolation:

**Ephemeral virtual environment.** The minimum bar — keeps the package off the system Python and makes cleanup trivial:

```{.bash filename="$ (local)"}
python -m venv /tmp/paw-sandbox && source /tmp/paw-sandbox/bin/activate
pip install programasweights \
    --extra-index-url https://pypi.programasweights.com/simple/
python -c "import programasweights as paw; print(paw.__version__)"
deactivate && rm -rf /tmp/paw-sandbox
```

This isolates the Python environment but not the filesystem or network — a malicious install script can still write to `~/.ssh` or make outbound calls.

**Docker container with network isolation.** For packages that run install scripts or make outbound calls, use a container with an explicit network policy:

```{.bash filename="$ (local)"}
# No outbound network access during install
docker run --rm --network none python:3.13-slim \
    pip install --no-index --find-links /wheels programasweights

# Or: allow network but observe it
docker run --rm python:3.13-slim sh -c \
    "pip install programasweights \
     --extra-index-url https://pypi.programasweights.com/simple/ \
     && python -c 'import programasweights'"
```

**`--no-build-isolation` + inspected build environment.** When the package uses a non-standard build backend, `--no-build-isolation` prevents pip from creating a temporary build environment. This forces the build to use your controlled environment but also means you can intercept and inspect all build steps. Use with an audited `pyproject.toml` or `setup.py`.

:::{.callout-note}
The goal is not to run every dependency in Docker forever — it is to have a reproducible record of what the package does on first install and first import before it touches production systems or CI secrets.

:::

### Private PyPI Index Trust

Packages served from a non-standard index (`--extra-index-url`, `--index-url`) introduce trust questions that do not apply to `pypi.org`. The threat model has two components:

**Dependency confusion attacks.** When both `pypi.org` and a private index are configured, `pip` and `uv` by default check *both* and install the highest version found on *either*. An attacker who knows the internal package name can publish a higher-versioned package with the same name to `pypi.org`, causing it to be installed instead of the private one. Mitigate by:

```{.bash filename="$ (local)"}
# uv: restrict a package to only the private index
uv add programasweights \
    --index https://pypi.programasweights.com/simple/ \
    --index-strategy unsafe-first-match  # never fall back to PyPI for this package
```

Or declare the index in `pyproject.toml` with explicit package pinning so the resolver never consults `pypi.org` for that name:

```{.toml filename="pyproject.toml"}
[[tool.uv.index]]
name = "programasweights"
url = "https://pypi.programasweights.com/simple/"
explicit = true   # only used for packages explicitly assigned to it

[tool.uv.sources]
programasweights = { index = "programasweights" }
```

With `explicit = true`, `uv` will not use this index for any package except those listed under `[tool.uv.sources]`. The dependency confusion attack surface is eliminated.

**Index server trust.** A private PyPI index is a web server you do not control. Before adding it:

- Verify the domain is owned by the vendor (WHOIS, GitHub org links, documentation cross-references).
- Check that the index URL is `https://` — plaintext HTTP allows MITM injection of malicious packages.
- Confirm `uv.lock` records SHA-256 hashes for packages fetched from the private index — the same integrity guarantee that applies to `pypi.org` packages applies here.

```{.bash filename="$ (local)"}
# Verify the hash recorded in uv.lock for a private-index package
grep -A5 'name = "programasweights"' uv.lock
```

If `uv.lock` does not record hashes for the private-index package (possible with some non-standard server configurations), do not use `--frozen` in CI — the hash verification step is silently skipped for that package.

:::{.callout-caution}
The dependency confusion attack is the highest-impact vector for private index packages. Always use `explicit = true` in `[tool.uv.index]` so the private index is never consulted for packages that should come from PyPI, and vice versa.

:::

## Appendix: Recovering from a Leaked Secret

If a secret is committed and pushed, the steps are (1) rotate the credential immediately — before anything else — and (2) assume it was seen. GitHub indexes commit history; forks preserve deleted content; cached copies may exist in search engines or scanning tools. History rewriting does not undo exposure retroactively.

**Rotation is the priority.** Removing the secret from history without rotating first leaves the window of exposure open while history rewriting is in progress. Rotate the key, then clean up history.

**Purging from git history.** `git filter-repo` is the modern replacement for `BFG Repo Cleaner` and `git filter-branch`. It rewrites all reachable commits:

```{.bash filename="$ (local)"}
# Install
pip install git-filter-repo

# Remove a specific file from all history
git filter-repo --path secrets.txt --invert-paths

# Replace a specific secret value with a placeholder in all history
git filter-repo --replace-text <(echo "sk-proj-ACTUAL_KEY==>REDACTED")

# Force-push all refs after rewrite
git push --force --all
git push --force --tags
```

After a force-push, all collaborators must re-clone — their local history is now incompatible with the rewritten remote. Communicate this before proceeding.

**GitHub secret scanning.** GitHub's Security tab (available free on public repositories) shows secret scanning alerts. Check it after a suspected leak to see what GitHub itself detected — this confirms whether the exposure was indexed.

:::{.callout-important}
Force-pushing does not immediately remove the commit from GitHub's servers. GitHub caches commits for a period after deletion. For sensitive credentials, contact GitHub Support after the force-push to request a cache purge.

:::

---

■